# Доходность «закрытие → закрытие» и окно новостей «9:30 → 9:30»

Третья постановка курсовой. Как и в `close_to_close.ipynb`, предсказываем доходность за сутки по закрытиям,
$r_t=100\ln(C_t/C_{t-1})$. Но новости для дня $t$ берём **в окне от 9:30 вчера до 9:30 сегодня** — двумя способами:
одним общим набором признаков (**один канал**) и отдельно для двух частей окна (**два канала**).

```
 9:30 t−1 ── сессия t−1 ── 16:00 ── вечер ── ночь ── утро ── 9:30 t ─────── сессия t ─────── 16:00 t
 |──── канал «сессия t−1» ────|────── канал «вечер, ночь, утро» ──────|
 |───────────────────── окно новостей «9:30 → 9:30» ──────────────────|
                              |───────────────────── доходность r_t ───────────────────────────|
                              |────────── ночной гэп ─────────|───────── сессия t ─────────────|
```

### Важно: где здесь прогноз, а где объяснение

Окно новостей и доходность **пересекаются** — с 16:00 $t-1$ до 9:30 $t$. Доходность состоит из двух частей:

* **ночной гэп** (закрытие $t-1$ → открытие $t$) происходит *одновременно* с вечерними, ночными и утренними новостями.
  Связь здесь — **объяснение**, а не прогноз: к моменту, когда новости окна известны (9:30), гэп уже случился;
* **сессия** (открытие $t$ → закрытие $t$) начинается после 9:30, когда все новости окна уже вышли. Это **настоящий
  прогноз**: позицию можно открыть на открытии и закрыть на закрытии.

Поэтому для $r_t$ целиком «предсказание» по окну «9:30 → 9:30» нельзя считать торгуемым прогнозом — часть ответа
(гэп) происходит одновременно с новостями. Тетрадка показывает это явно: модели строятся и для $r_t$, и отдельно для
гэпа и сессии. Для сравнения есть **эталон** — честный прогноз из `close_to_close.ipynb`: новости только до закрытия
$t-1$.

### Три схемы

| схема | статьи сессии $t-1$ (9:30–16:00 $t-1$) | статьи вечера, ночи и утра (16:00 $t-1$ → 9:30 $t$) | для $r_t$ это |
|---|---|---|---|
| **эталон**: сутки до закрытия $t-1$ | → день $t$ | → день $t+1$ (выходят после закрытия $t-1$) | чистый прогноз |
| **9:30 → 9:30, один канал** | → день $t$ | → день $t$, вместе с сессией $t-1$ | прогноз сессии + объяснение гэпа |
| **9:30 → 9:30, два канала** | → день $t$, свои признаки | → день $t$, свои признаки | то же, но части окна не смешиваются |

Разница между эталоном и «9:30 → 9:30» — только в утренних статьях: эталон отдаёт их следующему дню, схема
«9:30 → 9:30» — этому. Поэтому сравнение «эталон ↔ один канал» показывает, сколько добавляют вечерние, ночные и
утренние новости, а «один ↔ два канала» — нужно ли разделять части окна.

### Что в тетрадке

| № | раздел |
|---|---|
| 1 | данные (как в `close_to_close.ipynb`) |
| 2 | эмбеддинги по схемам и каналам |
| 3–4 | бак, обучение, метрики (как в `close_to_close.ipynb`) |
| 5 | главная таблица: $r_t$ — три схемы × эмбеддинги × две формы модели |
| 6 | части доходности: гэп (объяснение) и сессия (прогноз) |
| 7 | тест «новости из будущего» |
| 8 | стратегия на сессии — единственная торгуемая часть |
| 9 | итоги |

In [ ]:
import hashlib
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.signal import lfilter
from scipy.stats import norm, spearmanr
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from scripts import config as C
from scripts.align import session_open_utc
from scripts.news import article_text

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3, "figure.figsize": (11, 3.6)})

# ---------- что и на каком периоде (как в первой тетрадке: кэш SBERT общий) ----------
TICKER = "SPY"
START, END = "2008-01-01", "2023-04-30"
TRAIN_END, VAL_END = "2016-12-31", "2019-12-31"
# ---------- модель ----------
HALF_LIVES = [0, 1, 2, 3, 5, 7, 14, 30, 60]
ALPHAS = [10, 100, 1000, 10000]      # для знака доходности сигнал очень слабый — нужны и сильные штрафы
D_EMB = 8
Z_WINDOW = 252
WARMUP = 252
USE_SBERT = True
SBERT_DEVICE = None                  # None — автоматически: cuda / mps / cpu
SBERT_BATCH = 256
# ---------- проверки ----------
COST_BP = 1.0                        # издержки стратегии: базисных пунктов за единицу оборота
SEED = 0

def make_rng(*keys):
    return np.random.default_rng([SEED, *keys])

---
# 1. Данные

Те же данные и тот же период, что в `close_to_close.ipynb`: доходность «закрытие → закрытие» в процентах и её части —
ночной гэп и сессия ($r_t$ = гэп + сессия). Новости — финансовые статьи; дальше их делим на статьи **до открытия**
своего торгового дня (вечер, ночь, утро) и статьи **во время сессии**.

In [ ]:
prices = pd.read_parquet(C.PROCESSED_FILES["prices"])
prices["date"] = pd.to_datetime(prices["date"])

NEWS_COLS = ["pub_time_utc", "trading_date", "is_finance", "flag_source", "headline", "abstract", "lead_paragraph"]
try:
    news = pd.read_parquet(C.PROCESSED_FILES["news"], columns=NEWS_COLS,
                           filters=[("trading_date", ">=", pd.Timestamp(START))])
except Exception:
    news = pd.read_parquet(C.PROCESSED_FILES["news"], columns=NEWS_COLS)
news["trading_date"] = pd.to_datetime(news["trading_date"])
news = news[news["trading_date"] >= START]
end = pd.Timestamp(END)
guessed = news.loc[news["flag_source"] == "text_model", "trading_date"]
if len(guessed):
    end = min(end, guessed.min() - pd.Timedelta(days=1))

def ohlc(ticker):
    p = prices[prices["ticker"] == ticker].set_index("date").sort_index()
    f = p["adj_close"] / p["close"]
    return pd.DataFrame({"open": p["open"] * f, "high": p["high"] * f, "low": p["low"] * f, "close": p["adj_close"]})

full = ohlc(TICKER)
dates = full.loc[START:end].index
gap_days = dates.to_series().diff().dt.days.fillna(1).to_numpy()
masks = {"train": np.asarray(dates <= TRAIN_END),
         "val": np.asarray((dates > TRAIN_END) & (dates <= VAL_END)),
         "test": np.asarray(dates > VAL_END)}

# доходности в процентах; всё на полной истории, потом режем по датам
full["r"] = 100 * np.log(full["close"]).diff()                       # закрытие → закрытие
full["gap"] = 100 * np.log(full["open"] / full["close"].shift(1))    # ночь: закрытие t−1 → открытие t
full["sess"] = 100 * np.log(full["close"] / full["open"])            # сессия: открытие → закрытие
full["abs_y"] = np.log(full["r"].abs() + 0.01)                       # log|r| — мера волатильности
M = full.reindex(dates)
r = M["r"].to_numpy()

vix = prices[prices["ticker"] == "^VIX"].set_index("date")["close"].sort_index()
M["log_vix_lag"] = np.log(vix).shift(1).reindex(dates)              # VIX на закрытии t−1

for k, m in masks.items():
    print(f"{k:5s} {dates[m][0].date()} → {dates[m][-1].date()}  {m.sum()} дней")
print(f"r_t: среднее {np.nanmean(r):+.3f}%, ст. откл. {np.nanstd(r):.2f}%, доля дней роста {np.nanmean(r > 0):.1%}")
print(f"проверка: r = гэп + сессия, макс. расхождение {np.nanmax(np.abs(M['r'] - M['gap'] - M['sess'])):.2e}")

In [ ]:
open_utc = pd.Series(session_open_utc(dates).tz_convert("UTC").tz_localize(None), index=dates)
news = news[news["is_finance"] & news["trading_date"].isin(dates)].copy()
pub = news["pub_time_utc"].dt.tz_convert("UTC").dt.tz_localize(None)
news["pre_open"] = (pub <= news["trading_date"].map(open_utc)).to_numpy()

# две группы — в том же порядке, что в первой тетрадке, чтобы векторы SBERT взялись из общего кэша
fin_pre = news[news["pre_open"]]
fin_rest = news[~news["pre_open"]]

hour = news["pub_time_utc"].dt.tz_convert("America/New_York")
hh = hour.dt.hour + hour.dt.minute / 60
same_day = (hour.dt.tz_localize(None).dt.normalize() == news["trading_date"]).to_numpy()
part = np.where(~same_day, "вечер t−1 и выходные", np.where(hh <= 9.5, "ночь и утро", "сессия t"))
display((pd.Series(part).value_counts(normalize=True) * 100).round(1).to_frame("% статей окна"))
print(f"финансовых статей: {len(news):,}, в среднем {len(news) / len(dates):.1f} в торговый день")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
lags = range(1, 21)
ax[0].bar(list(lags), [pd.Series(r)[np.isfinite(r)].autocorr(l) for l in lags])
ax[0].set_title("автокорреляция r_t: почти ноль — знак плохо предсказуем по прошлому")
ax[1].plot(dates, M["r"].rolling(63).std(), lw=1)
ax[1].set_title("волатильность r_t (скользящее ст. откл., 63 дня): меняется сильно")
plt.tight_layout(); plt.show()

---
# 2. Эмбеддинги по схемам и каналам

Каждая статья относится к одной из двух групп своего торгового дня $D$ (`trading_date`):

* **до открытия** (`pre`): вышла между закрытием $D-1$ и 9:30 $D$;
* **во время сессии** (`rest`): вышла между 9:30 и 16:00 $D$.

Схема говорит, к какому дню прогноза $t$ отнести статью каждой группы: к самому дню $D$ (**same**) или к следующему
торговому дню (**next**). Так все три схемы задаются одной таблицей:

| схема | канал | `pre` (вечер, ночь, утро дня $D$) | `rest` (сессия дня $D$) |
|---|---|---|---|
| эталон | один | next | next |
| 9:30 → 9:30, один канал | один | same | next |
| 9:30 → 9:30, два канала | «вечер, ночь, утро» | same | — |
| | «сессия $t-1$» | — | next |

Эмбеддинги те же четыре; их функции и кэш SBERT — как в `close_to_close.ipynb`. Каждый признак очищается скользящим
z-score по прошлому году.

In [ ]:
tr = masks["train"]

NEG = """loss losses lose losing fall falls fell falling drop drops dropped decline declines declined plunge plunged
slump slumped crash crisis fear fears worry worries worried recession default defaults bankruptcy bankrupt layoffs cuts weak
weakness slowdown downturn selloff turmoil panic collapse fail failed failure threat threatens trouble troubled fraud
debt deficit unemployment""".split()
POS = """gain gains gained rise rises rose rising rally rallied surge surged jump jumped growth grow grew strong strength
recovery recover rebound boost improve improved improvement profit profits profitable optimism optimistic upbeat beat""".split()
UNC = """uncertain uncertainty unclear volatile volatility may might could possibly unsettled doubt doubts concern concerns
concerned unpredictable turbulence risk risks risky""".split()

def lexicon_features(texts):
    vocab = sorted(set(NEG + POS + UNC))
    col = {w: i for i, w in enumerate(vocab)}
    cnt = CountVectorizer(vocabulary=vocab, token_pattern=r"(?u)\b\w+\b").transform(texts)
    n_words = texts.str.split().str.len().fillna(0).clip(lower=1).to_numpy()
    rate = lambda ws: 1000 * np.asarray(cnt[:, [col[w] for w in set(ws)]].sum(axis=1)).ravel() / n_words
    return pd.DataFrame({"neg_rate": rate(NEG), "pos_rate": rate(POS), "unc_rate": rate(UNC)}, index=texts.index)

def tfidf_features(texts, d=D_EMB):
    vec = TfidfVectorizer(min_df=5, max_df=0.5, max_features=30_000, ngram_range=(1, 2),
                          sublinear_tf=True, stop_words="english").fit(texts[tr])
    Xs = vec.transform(texts)
    svd = TruncatedSVD(n_components=d, algorithm="arpack", random_state=SEED).fit(Xs[tr])
    return pd.DataFrame(svd.transform(Xs), index=dates, columns=[f"svd{i + 1}" for i in range(d)])

def past_zscore(df, window=Z_WINDOW, minp=60, clip=5):
    mu = df.rolling(window, min_periods=minp).mean().shift(1)
    sd = df.rolling(window, min_periods=minp).std().shift(1)
    return ((df - mu) / sd).clip(-clip, clip).fillna(0.0)

# ---------- SBERT: тот же кэш, что в первой тетрадке ----------
SBERT_MODEL = None
def sbert_device():
    if SBERT_DEVICE:
        return SBERT_DEVICE
    try:
        import torch
    except ImportError:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    return "mps" if mps is not None and mps.is_available() else "cpu"

def sbert_encode(arts):
    global SBERT_MODEL
    key = hashlib.md5("\n".join(arts).encode("utf-8")).hexdigest()[:12]
    cache = C.EMBED_DIR / f"preopen_minilm_{key}.npz"
    if cache.exists():
        return np.load(cache)["emb"]
    from sentence_transformers import SentenceTransformer
    dev = sbert_device()
    if SBERT_MODEL is None:
        SBERT_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=dev)
    print(f"SBERT: кодируем {len(arts):,} статей на {dev}")
    emb = np.asarray(SBERT_MODEL.encode(list(arts), batch_size=SBERT_BATCH if dev != "cpu" else 64,
                                        show_progress_bar=True, normalize_embeddings=True), dtype=np.float32)
    C.EMBED_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(cache.stem + ".tmp.npz")
    np.savez_compressed(tmp, emb=emb)
    tmp.replace(cache)
    return emb

GROUPS = {"pre": fin_pre, "rest": fin_rest}
SB_EMB = {}
if USE_SBERT:
    try:
        for g, df in GROUPS.items():
            SB_EMB[g] = sbert_encode(article_text(df).to_numpy())
    except ImportError:
        print("sentence-transformers не установлен — SBERT пропускаем")
        SB_EMB = {}


def next_trading_day(td):
    # для каждой даты из календаря — следующий торговый день (NaT для последнего)
    pos = np.searchsorted(dates.to_numpy(), td)
    out = np.full(len(td), np.datetime64("NaT"), dtype="datetime64[ns]")
    inside = pos + 1 < len(dates)
    out[inside] = dates.to_numpy()[pos[inside] + 1]
    return out

TD = {g: df["trading_date"].to_numpy().astype("datetime64[ns]") for g, df in GROUPS.items()}
KEYS = {"same": TD, "next": {g: next_trading_day(td) for g, td in TD.items()}}

SCHEMES = {
    "эталон: до закрытия t−1": [{"pre": "next", "rest": "next"}],
    "9:30 → 9:30, один канал": [{"pre": "same", "rest": "next"}],
    "9:30 → 9:30, два канала": [{"pre": "same"}, {"rest": "next"}],
}
KINDS = ["volume", "lexicon", "tfidf"] + (["sbert"] if SB_EMB else [])

_EPS_CACHE = {}
def build_eps(spec):
    # ε по дням прогноза для одного канала: spec = {группа: "same" | "next"}
    key_spec = tuple(sorted(spec.items()))
    if key_spec in _EPS_CACHE:
        return _EPS_CACHE[key_spec]
    dfs = [GROUPS[g] for g in spec]
    keys = np.concatenate([KEYS[kt][g] for g, kt in spec.items()])
    allart = pd.concat(dfs)
    order = np.argsort(allart.index.to_numpy(), kind="stable")      # исходный порядок статей датасета
    allart, keys = allart.iloc[order], keys[order]
    okk = ~np.isnat(keys)
    texts = article_text(allart[okk]).groupby(keys[okk]).agg("\n".join).reindex(dates).fillna("")
    n = pd.Series(keys[okk]).value_counts().reindex(dates).fillna(0).to_numpy()
    raw = {"volume": pd.DataFrame({"log_n": np.log1p(n / gap_days)}, index=dates),
           "lexicon": lexicon_features(texts),
           "tfidf": tfidf_features(texts)}
    if SB_EMB:
        S, cnt = 0, 0
        for g, kt in spec.items():
            k = KEYS[kt][g]
            ok_g = ~np.isnat(k)
            S = S + pd.DataFrame(SB_EMB[g][ok_g]).groupby(k[ok_g]).sum().reindex(dates).fillna(0.0)
            cnt = cnt + pd.Series(k[ok_g]).value_counts().reindex(dates).fillna(0)
        day = S.div(cnt.replace(0, np.nan), axis=0)
        day = day.fillna(day.loc[tr].mean())
        pca = PCA(n_components=D_EMB, svd_solver="full").fit(day.loc[tr])
        raw["sbert"] = pd.DataFrame(pca.transform(day), index=dates, columns=[f"sbert{i + 1}" for i in range(D_EMB)])
    out = {k: past_zscore(v).to_numpy() for k, v in raw.items()}
    _EPS_CACHE[key_spec] = out
    return out

# признаки каждой схемы: каналы ставятся рядом
FEATS = {sch: {kind: np.hstack([build_eps(spec)[kind] for spec in chans]) for kind in KINDS}
         for sch, chans in SCHEMES.items()}
for sch, f in FEATS.items():
    print(f"{sch:26s}", {k: v.shape[1] for k, v in f.items()})

def mean_articles(spec):
    keys = np.concatenate([KEYS[kt][g] for g, kt in spec.items()])
    return pd.Series(keys[~np.isnat(keys)]).value_counts().reindex(dates).fillna(0).mean()
print("\nстатей в окне в среднем за день:")
for sch, chans in SCHEMES.items():
    print(f"  {sch:26s}", " + ".join(f"{mean_articles(c):.1f}" for c in chans))

**Проверка схем.** Для любого дня $t$ в эталоне и в схеме «9:30 → 9:30» одни и те же статьи сессии $t-1$; различаются
только утренние: эталон отдаёт статьи до 9:30 дня $t$ дню $t+1$, схема «9:30 → 9:30» — дню $t$. Каждая статья попадает
ровно в одно окно (кроме последнего дня выборки, где следующего дня нет).

In [ ]:
def ou_state(eps, gaps, half_life):
    # S_t = A^{Δt} S_{t-1} + ε_t, A = diag(0.5^(1/h)); быстро через календарную сетку и lfilter
    E = np.asarray(eps, dtype=float)
    hl = np.broadcast_to(np.asarray(half_life, dtype=float), (E.shape[1],))
    pos = np.r_[0, np.cumsum(np.round(np.asarray(gaps, dtype=float)[1:]).astype(int))]
    G = np.zeros((pos[-1] + 1, E.shape[1]))
    G[pos] = E
    X = E.copy()
    for h in np.unique(hl[hl > 0]):
        cols = hl == h
        X[:, cols] = lfilter([1.0], [1.0, -0.5 ** (1.0 / h)], G[:, cols], axis=0)[pos]
    return X

def shift_rows(X, k):
    # сдвиг вниз на k строк: строка t получает значение строки t−k (k < 0 — вверх, «из будущего»)
    X = np.asarray(X, dtype=float)
    out = np.full_like(X, np.nan)
    if k > 0:
        out[k:] = X[:-k]
    elif k < 0:
        out[:k] = X[-k:]
    else:
        out = X.copy()
    return out

def news_design(E, h, mode, lag=1):
    # признаки для r_t: lag=1 — прогноз (новости до закрытия t−1), lag=0 — объяснение (новости дня t)
    S = ou_state(E, gap_days, h)
    F = S if mode == "уровень" else S - shift_rows(S, 1)
    return shift_rows(F, lag)

In [ ]:
def fit_predict(F, y, fit_mask, alpha):
    F, y = np.asarray(F, dtype=float), np.asarray(y, dtype=float)
    ok_ = np.isfinite(F).all(axis=1) & np.isfinite(y)
    m = fit_mask & ok_
    mu, sd = F[m].mean(axis=0), F[m].std(axis=0)
    sd = np.where(sd > 0, sd, 1.0)
    Z = (F[m] - mu) / sd
    y0 = y[m].mean()
    g = np.linalg.solve(Z.T @ Z + alpha * np.eye(Z.shape[1]), Z.T @ (y[m] - y0))
    pred = np.full(len(y), np.nan)
    pred[ok_] = y0 + ((F[ok_] - mu) / sd) @ g
    return pred, g

def mse(y, p, m):
    return np.mean((y[m] - p[m]) ** 2)

def select(make_F, y, okv, half_lives=HALF_LIVES, alphas=ALPHAS):
    trm, vam = masks["train"] & okv, masks["val"] & okv
    best = None
    for h in half_lives:
        F = make_F(h)
        fin = np.isfinite(F).all(axis=1)
        for a in alphas:
            p = fit_predict(F, y, trm & fin, a)[0]
            v = vam & fin
            s = np.mean((y[v] - p[v]) ** 2)
            if best is None or s < best[0]:
                best = (s, h, a)
    r2_val = 1 - best[0] / np.mean((y[vam] - y[trm].mean()) ** 2)
    return {"half_life": best[1], "alpha": best[2], "R² val": r2_val}

# ---------- метрики ----------
def oos_r2(y, p, m, ref):
    return 1 - np.sum((y[m] - p[m]) ** 2) / np.sum((y[m] - ref) ** 2)

def newey_west_se(x, lags=10):
    x = np.asarray(x, float) - np.mean(x)
    n = len(x)
    s = x @ x / n
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * (x[l:] @ x[:-l]) / n
    return np.sqrt(max(s, 0) / n)

def clark_west(y, p_base, p_new, lags=10):
    f = (y - p_base) ** 2 - (y - p_new) ** 2 + (p_base - p_new) ** 2
    t = f.mean() / newey_west_se(f, lags)
    return t, 1 - norm.cdf(t)

def pesaran_timmermann(y, f):
    # доля угаданных знаков и односторонний тест «лучше, чем при независимости прогноза и факта»
    y, f = np.asarray(y) > 0, np.asarray(f) > 0
    n = len(y)
    py, px = y.mean(), f.mean()
    p_hat = (y == f).mean()
    p_star = py * px + (1 - py) * (1 - px)
    v_hat = p_star * (1 - p_star) / n
    v_star = ((2 * py - 1) ** 2 * px * (1 - px) + (2 * px - 1) ** 2 * py * (1 - py)) / n \
             + 4 * py * px * (1 - py) * (1 - px) / n ** 2
    d = v_hat - v_star
    if d <= 0:
        return p_hat, np.nan
    return p_hat, 1 - norm.cdf((p_hat - p_star) / np.sqrt(d))

def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(running, 1.0)
    return adj

---
# 5. Главная таблица: доходность $r_t$

Все признаки уже приписаны **дню прогноза** $t$ (окно заканчивается до 9:30 $t$ или до закрытия $t-1$), поэтому в модели
используется сдвиг `lag=0`: бак $S_t$ включает новости окна дня $t$. Две формы, как в `close_to_close.ipynb`:
**разность** (буквальный план, $r_t\approx g_0+\langle g, S_t-S_{t-1}\rangle$) и **уровень** ($r_t\approx g_0+\langle g, S_t\rangle$).
Метрики на тесте: OOS $R^2$ против исторического среднего, CW p (с Холмом по всей таблице), доля угаданных знаков
с тестом Песарана–Тиммерманна.

Напоминание: для схем «9:30 → 9:30» результат по $r_t$ — смесь объяснения гэпа и прогноза сессии; раздел 6 их разделит.

In [ ]:
ok_r = np.isfinite(r) & (np.arange(len(dates)) >= WARMUP)
TV = (masks["train"] | masks["val"]) & ok_r
TST = masks["test"] & ok_r

def evaluate(E, mode, y, okv):
    s = select(lambda h: news_design(E, h, mode, 0), y, okv)
    F = news_design(E, s["half_life"], mode, 0)
    tv, te = (masks["train"] | masks["val"]) & okv, masks["test"] & okv
    fin = np.isfinite(F).all(axis=1)
    p = fit_predict(F, y, tv & fin, s["alpha"])[0]
    ref = y[tv].mean()
    te = te & fin
    hit, pt_p = pesaran_timmermann(y[te] - ref, p[te] - ref)
    return {"h": s["half_life"], "α": s["alpha"], "R² val": s["R² val"],
            "OOS R² test, %": 100 * oos_r2(y, p, te, ref),
            "CW p": clark_west(y[te], np.full(te.sum(), ref), p[te])[1],
            "знаки угаданы": hit, "PT p": pt_p}, p

rows, PRED = [], {}
for sch in SCHEMES:
    for kind in KINDS:
        for mode in ("разность", "уровень"):
            res, p = evaluate(FEATS[sch][kind], mode, r, ok_r)
            rows.append({"схема": sch, "эмбеддинг": kind, "форма": mode, **res})
            PRED[(sch, kind, mode)] = p
main = pd.DataFrame(rows)
main["CW p (Холм)"] = holm(main["CW p"])
display(main.set_index(["схема", "эмбеддинг", "форма"]).round(4))

piv = main.pivot_table(index=["эмбеддинг", "форма"], columns="схема", values="OOS R² test, %").reindex(columns=list(SCHEMES))
display(piv.round(3))
fig, ax = plt.subplots(figsize=(11, 3.8))
xs = np.arange(len(piv)); w = 0.27
for i, sch in enumerate(SCHEMES):
    ax.bar(xs + (i - 1) * w, piv[sch], width=w, label=sch)
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(xs); ax.set_xticklabels([f"{a}\n{b}" for a, b in piv.index], fontsize=8)
ax.set_ylabel("OOS R² test, %"); ax.set_title("Доходность закрытие → закрытие: три схемы окна новостей"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Как читать.**

* **Эталон** — честный прогноз. Его OOS $R^2$ — та планка, которую можно назвать предсказуемостью доходности.
* Если **«9:30 → 9:30» заметно выше эталона**, это почти наверняка за счёт ночного гэпа: новости вечера, ночи и утра
  объясняют движение, которое происходит одновременно с ними. Раздел 6 это проверит.
* **Один канал против двух**: если два канала лучше, части окна несут разную информацию (например, утренние новости
  связаны с гэпом, а дневные — нет), и смешивать их вредно.

---
# 6. Части доходности: гэп и сессия

Те же схемы и эмбеддинги, но цели — по отдельности:

* **ночной гэп** — для схем «9:30 → 9:30» это **объяснение** (новости и гэп одновременны); для эталона — прогноз;
* **сессия $t$** — для всех схем **прогноз** (все новости окна вышли до 9:30 $t$).

Форму модели (разность / уровень) берём лучшую по валидации для каждой пары «схема × эмбеддинг».

In [ ]:
TARGETS = {"ночной гэп": M["gap"].to_numpy(), "сессия t": M["sess"].to_numpy()}
rows = []
for tgt, yv in TARGETS.items():
    okv = np.isfinite(yv) & (np.arange(len(dates)) >= WARMUP)
    for sch in SCHEMES:
        for kind in KINDS:
            best = None
            for mode in ("разность", "уровень"):
                res, p = evaluate(FEATS[sch][kind], mode, yv, okv)
                if best is None or res["R² val"] > best[0]["R² val"]:
                    best = (res, mode, p)
            res, mode, p = best
            rows.append({"цель": tgt, "схема": sch, "эмбеддинг": kind, "форма": mode, **res})
            PRED[(sch, kind, "лучшая", tgt)] = p
parts = pd.DataFrame(rows)
parts["CW p (Холм)"] = holm(parts["CW p"])
display(parts.set_index(["цель", "схема", "эмбеддинг"]).round(4))
display(parts.pivot_table(index=["цель", "эмбеддинг"], columns="схема", values="OOS R² test, %")
        .reindex(columns=list(SCHEMES)).round(3))

**Как читать.**

* **Гэп, схемы «9:30 → 9:30»**: высокий OOS $R^2$ — ожидаемо, это одновременная связь. Например, вечером вышел
  отчёт компании или новости из Азии, и рынок открылся с гэпом. Это интересно как *объяснение* того, что двигает
  цену ночью, но это не прогноз.
* **Гэп, эталон**: новости до закрытия $t-1$ — настоящий прогноз гэпа; обычно около нуля.
* **Сессия, все схемы**: это и есть честная проверка, есть ли у окна «9:30 → 9:30» прогнозная сила. Если здесь
  «9:30 → 9:30» лучше эталона, утренние новости действительно помогают предсказать день — тогда торговать имеет смысл
  с открытия (раздел 8).

---
# 7. Тест «новости из будущего»

Возьмём окно «9:30 → 9:30», заканчивающееся **на день позже** (в 9:30 $t+1$), и попробуем «объяснить» им $r_t$. В это окно
попадают статьи сессии дня $t$ — в том числе сводки о том, как прошли торги. Причинного влияния здесь быть не может:
эти статьи вышли **после** того, как доходность частично или полностью сложилась. Если связь сильная, значит, тексты
пересказывают движение, и это мерило того, насколько «объяснение» в разделе 6 может быть пересказом.

Без памяти ($h=0$), форма «уровень».

In [ ]:
rows = []
for kind in KINDS:
    E = FEATS["9:30 → 9:30, один канал"][kind]
    for name, Ek in (("окно до 9:30 t (схема)", E), ("окно до 9:30 t+1 (из будущего)", shift_rows(E, -1))):
        for tgt, yv in (("r_t", r), ("ночной гэп", M["gap"].to_numpy()), ("сессия t", M["sess"].to_numpy())):
            okv = np.isfinite(yv) & (np.arange(len(dates)) >= WARMUP)
            s = select(lambda h: Ek, yv, okv, half_lives=[0])
            fin = np.isfinite(Ek).all(axis=1)
            tv, te = (masks["train"] | masks["val"]) & okv & fin, masks["test"] & okv & fin
            p = fit_predict(Ek, yv, tv, s["alpha"])[0]
            rows.append({"эмбеддинг": kind, "окно": name, "цель": tgt,
                         "OOS R² test, %": 100 * oos_r2(yv, p, te, yv[tv].mean())})
lead = pd.DataFrame(rows).pivot_table(index=["эмбеддинг", "окно"], columns="цель", values="OOS R² test, %")
display(lead[["r_t", "ночной гэп", "сессия t"]].round(3))

**Как читать.** Строка «из будущего» с высоким $R^2$ для сессии $t$ — прямое доказательство пересказа: статьи о торгах
дня $t$ «объясняют» доходность дня $t$, потому что написаны о ней. Если для гэпа и для сессии «из будущего» около нуля,
а схема даёт заметную связь с гэпом, — новости вечера и ночи действительно двигают цену, а не пересказывают её.

---
# 8. Стратегия на сессии — единственная торгуемая часть

Новости окна «9:30 → 9:30» известны к открытию, поэтому честная стратегия — **открыть позицию на открытии $t$ и закрыть
на закрытии $t$**: long, если прогноз доходности сессии выше среднего, иначе short (или деньги). Сравниваем с:

* «покупать каждую сессию» (long на открытии, продажа на закрытии каждый день, с теми же издержками);
* эталонной стратегией «закрытие → закрытие» по новостям до закрытия $t-1$ (лучшая по валидации модель $r_t$ из
  эталонной схемы).

Модель для сессии — лучшая по валидации среди схем «9:30 → 9:30». Издержки — `COST_BP` б.п. за единицу оборота
(у стратегий на сессии оборот выше: позиция открывается и закрывается каждый день).

In [ ]:
def strategy(rp, w, cost_bp=COST_BP):
    R = np.expm1(rp / 100)
    turn = np.abs(np.diff(np.r_[0.0, w]))
    s = w * R - turn * cost_bp / 1e4
    eq = np.cumprod(1 + s)
    return {"доходность, % годовых": 100 * (eq[-1] ** (252 / len(s)) - 1),
            "волатильность, %": 100 * s.std() * np.sqrt(252),
            "Шарп": s.mean() / s.std() * np.sqrt(252) if s.std() > 0 else np.nan,
            "макс. просадка, %": 100 * (eq / np.maximum.accumulate(eq) - 1).min(),
            "оборот в год": turn.sum() * 252 / len(s)}, eq


sess = M["sess"].to_numpy()
cand = parts[(parts["цель"] == "сессия t") & parts["схема"].str.startswith("9:30")]
bs = cand.loc[cand["R² val"].idxmax()]
p_sess = PRED[(bs["схема"], bs["эмбеддинг"], "лучшая", "сессия t")]
okv = np.isfinite(sess) & (np.arange(len(dates)) >= WARMUP)
ref_s = sess[(masks["train"] | masks["val"]) & okv].mean()
te_s = masks["test"] & okv & np.isfinite(p_sess)

cand0 = main[main["схема"] == "эталон: до закрытия t−1"]
b0 = cand0.loc[cand0["R² val"].idxmax()]
p0 = PRED[(b0["схема"], b0["эмбеддинг"], b0["форма"])]
te0 = TST & np.isfinite(p0)

def session_strategy(rp, w, cost_bp=COST_BP):
    # позиция открывается на открытии и закрывается на закрытии каждый день: издержки на вход и выход
    R = np.expm1(rp / 100)
    s_ = w * R - 2 * np.abs(w) * cost_bp / 1e4
    eq = np.cumprod(1 + s_)
    return {"доходность, % годовых": 100 * (eq[-1] ** (252 / len(s_)) - 1),
            "волатильность, %": 100 * s_.std() * np.sqrt(252),
            "Шарп": s_.mean() / s_.std() * np.sqrt(252) if s_.std() > 0 else np.nan,
            "макс. просадка, %": 100 * (eq / np.maximum.accumulate(eq) - 1).min(),
            "оборот в год": 2 * np.abs(w).mean() * 252}, eq

rows, curves = [], {}
for name, fn, rp, w, te in (
        ("сессия: покупать каждый день", session_strategy, sess, np.ones(te_s.sum()), te_s),
        (f"сессия: long/short ({bs['схема']}, {bs['эмбеддинг']})", session_strategy, sess,
         np.sign(p_sess[te_s] - ref_s), te_s),
        (f"сессия: long/cash ({bs['схема']}, {bs['эмбеддинг']})", session_strategy, sess,
         (p_sess[te_s] > ref_s).astype(float), te_s),
        ("закрытие→закрытие: купил и держи", strategy, r, np.ones(te0.sum()), te0),
        (f"закрытие→закрытие: long/short (эталон, {b0['эмбеддинг']})", strategy, r,
         np.sign(p0[te0] - r[TV].mean()), te0)):
    st, eq = fn(rp[te], w)
    rows.append({"стратегия": name, **st}); curves[name] = (dates[te], eq)
display(pd.DataFrame(rows).set_index("стратегия").round(3))

fig, ax = plt.subplots()
for name, (dd, eq) in curves.items():
    ax.plot(dd, eq, label=name, lw=1)
ax.set_title("тест: рост 1 доллара"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

**Как читать.** Если стратегия на сессии по новостям «9:30 → 9:30» обгоняет «покупать каждую сессию» после издержек —
у утренних новостей есть торгуемая прогнозная сила. Сравнение с эталонной стратегией «закрытие → закрытие» показывает,
что лучше: решать на закрытии по вчерашним новостям или на открытии по утренним. Это иллюстрация на одном инструменте,
без проскальзывания и налогов.

---
# 9. Итоги

In [ ]:
def best_of(df, **flt):
    d = df
    for k, v in flt.items():
        d = d[d[k] == v]
    return d.loc[d["R² val"].idxmax()]

print("Доходность r_t (закрытие → закрытие), лучшая по валидации модель в каждой схеме:")
for sch in SCHEMES:
    b = best_of(main, схема=sch)
    print(f"  {sch:26s} {b['эмбеддинг']:8s} {b['форма']:9s} OOS R² = {b['OOS R² test, %']:+.3f}%  "
          f"CW p (Холм) = {b['CW p (Холм)']:.3f}  знаки {b['знаки угаданы']:.1%}")
print("\nЧасти доходности, лучшая модель (OOS R² на тесте, %):")
for tgt in TARGETS:
    print(f"  {tgt}:", ", ".join(f"{sch}: {best_of(parts, цель=tgt, схема=sch)['OOS R² test, %']:+.3f}"
                                 for sch in SCHEMES))
print("\nТест «новости из будущего» (средний OOS R² по эмбеддингам, %):")
print(lead.groupby(level="окно").mean().round(3).to_string())

### Как читать итог

| что видим | что это значит |
|---|---|
| «9:30 → 9:30» сильно лучше эталона для $r_t$, но только за счёт гэпа | утренние новости *объясняют* ночное движение; прогнозом для $r_t$ это не является |
| «9:30 → 9:30» лучше эталона и для **сессии** | утренние новости помогают *предсказать* день — торговать с открытия имеет смысл |
| два канала лучше одного | части окна несут разную информацию: смешивать дневные и утренние новости вредно |
| «новости из будущего» сильно связаны с сессией $t$ | тексты пересказывают торги; одновременные связи надо читать осторожно |
| эталон около нуля | по новостям до закрытия доходность следующего дня не предсказать (как в `close_to_close.ipynb`) |

**Связь с другими тетрадками.** В `baseline_simple` утренние новости предсказывали *волатильность* сессии; здесь
проверяется *доходность*. В `close_to_close.ipynb` — чистый прогноз на закрытии; здесь — что меняется, если сдвинуть
конец окна на 9:30 и честно отделить объяснение гэпа от прогноза сессии.